In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

In [2]:
df = pd.read_csv("spotify_tracks.csv")

In [3]:
df.head()

,track_id,track_name,artist_name,year,popularity,artwork_url,album_name,acousticness,danceability,duration_ms,...,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,track_url,language
0,2r0ROhr7pRN4MXDMT1fEmd,"Leo Das Entry (From ""Leo"")",Anirudh Ravichander,2024,59,https://i.scdn.co/image/ab67616d0000b273ce9c65...,"Leo Das Entry (From ""Leo"")",0.0241,0.753,97297.0,...,8.0,0.1000,-5.994,0.0,0.1030,110.997,4.0,0.459,https://open.spotify.com/track/2r0ROhr7pRN4MXD...,Tamil
1,4I38e6Dg52a2o2a8i5Q5PW,AAO KILLELLE,"Anirudh Ravichander, Pravin Mani, Vaishali Sri...",2024,47,https://i.scdn.co/image/ab67616d0000b273be1b03...,AAO KILLELLE,0.0851,0.780,207369.0,...,10.0,0.0951,-5.674,0.0,0.0952,164.995,3.0,0.821,https://open.spotify.com/track/4I38e6Dg52a2o2a...,Tamil
2,59NoiRhnom3lTeRFaBzOev,Mayakiriye Sirikiriye - Orchestral EDM,"Anirudh Ravichander, Anivee, Alvin Bruno",2024,35,https://i.scdn.co/image/ab67616d0000b27334a1dd...,Mayakiriye Sirikiriye (Orchestral EDM),0.0311,0.457,82551.0,...,2.0,0.0831,-8.937,0.0,0.1530,169.996,4.0,0.598,https://open.spotify.com/track/59NoiRhnom3lTeR...,Tamil
3,5uUqRQd385pvLxC8JX3tXn,Scene Ah Scene Ah - Experimental EDM Mix,"Anirudh Ravichander, Bharath Sankar, Kabilan, ...",2024,24,https://i.scdn.co/image/ab67616d0000b27332e623...,Scene Ah Scene Ah (Experimental EDM Mix),0.2270,0.718,115831.0,...,7.0,0.1240,-11.104,1.0,0.4450,169.996,4.0,0.362,https://open.spotify.com/track/5uUqRQd385pvLxC...,Tamil
4,1KaBRg2xgNeCljmyxBH1mo,Gundellonaa X I Am A Disco Dancer - Mashup,"Anirudh Ravichander, Benny Dayal, Leon James, ...",2024,22,https://i.scdn.co/image/ab67616d0000b2735a59b6...,Gundellonaa X I Am a Disco Dancer (Mashup),0.0153,0.689,129621.0,...,7.0,0.3450,-9.637,1.0,0.1580,128.961,4.0,0.593,https://open.spotify.com/track/1KaBRg2xgNeCljm...,Tamil


In [4]:
df[df["language"] == 'English'].shape

(23392, 22)

In [5]:
df[df["language"] == 'Hindi'].shape

(5740, 22)

In [6]:
df.columns

Index(['track_id', 'track_name', 'artist_name', 'year', 'popularity',
       'artwork_url', 'album_name', 'acousticness', 'danceability',
       'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness',
       'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence',
       'track_url', 'language'],
      dtype='str')

In [10]:
feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]

print("Features used:")
for feature in feature_columns:
    print("-", feature)

Features used:
- acousticness
- danceability
- energy
- instrumentalness
- liveness
- loudness
- speechiness
- tempo
- valence


In [7]:
tar_eng = ["English"]
df_eng = df[df["language"].isin(tar_eng)]

In [11]:
df_eng = df_eng.drop_duplicates(
    subset="track_id"
).copy()

print("After removing duplicate track IDs:", df_eng.shape)

After removing duplicate track IDs: (23389, 22)


In [13]:
df_eng = df_eng.drop_duplicates(
    subset=["track_name", "artist_name"],
    keep="first"
).copy()

print("Dataset shape after removing track + artist duplicates:", df_eng.shape)

Dataset shape after removing track + artist duplicates: (14624, 22)


In [16]:
df_eng = df_eng.dropna(
    subset=feature_columns
).copy()

print("After removing missing feature values:", df_eng.shape)

After removing missing feature values: (14624, 22)


In [19]:
df_eng = df_eng.reset_index(drop=True)

In [20]:
X_eng = df_eng[feature_columns].copy()

print("Feature matrix shape:", X_eng.shape)
display(X_eng.head())

Feature matrix shape: (14624, 9)


,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence
0,0.1190,0.801,0.504,0.000000,0.132,-5.771,0.2590,119.971,0.821
1,0.4510,0.609,0.491,0.000000,0.111,-9.933,0.0945,99.186,0.718
2,0.0264,0.753,0.827,0.000582,0.358,-7.441,0.0467,127.998,0.512
3,0.5010,0.860,0.451,0.000000,0.355,-10.897,0.1740,119.806,0.360
4,0.5060,0.647,0.547,0.000000,0.216,-8.115,0.1850,179.677,0.793


In [24]:
scaler_eng = StandardScaler()

X_scaled_eng = scaler_eng.fit_transform(X_eng)

print("Scaled feature matrix shape:", X_scaled_eng.shape)

Scaled feature matrix shape: (14624, 9)


In [26]:
knn_eng = NearestNeighbors(
    n_neighbors=20,
    metric="cosine",
    algorithm="brute"
)

knn_eng.fit(X_scaled_eng)

print("KNN model for English created successfully.")

KNN model for English created successfully.


In [32]:
def recommend_from_features_english(
    acousticness,
    danceability,
    energy,
    instrumentalness,
    liveness,
    loudness,
    speechiness,
    tempo,
    valence,
    n_recommendations=5
):
    
    # Create a DataFrame containing the new song
    new_song = pd.DataFrame([{
        "acousticness": acousticness,
        "danceability": danceability,
        "energy": energy,
        "instrumentalness": instrumentalness,
        "liveness": liveness,
        "loudness": loudness,
        "speechiness": speechiness,
        "tempo": tempo,
        "valence": valence
    }])
    
    # Scale using the SAME scaler used during model preparation
    new_song_scaled = scaler_eng.transform(new_song[feature_columns])
    
    # Find nearest songs
    distances, indices = knn_eng.kneighbors(
        new_song_scaled,
        n_neighbors=n_recommendations
    )
    
    # Build result
    recommendations = df_eng.iloc[indices[0]].copy()
    
    # Convert cosine distance to cosine similarity
    recommendations["similarity"] = 1 - distances[0]
    
    # Select useful columns
    result_columns = [
        "track_id",
        "track_name",
        "artist_name",
        "album_name",
        "year",
        "language",
        "popularity",
        "similarity"
    ]
    
    return recommendations[result_columns].reset_index(drop=True)

In [33]:
recommendations = recommend_from_features_english(
    acousticness=0.79,
    danceability=0.535,
    energy=0.462,
    instrumentalness=2.07e-06,
    liveness=0.114,
    loudness=-11.291,
    speechiness=0.0296,
    tempo=145.959,
    valence=0.23,
    n_recommendations=10
)

display(recommendations)

,track_id,track_name,artist_name,album_name,year,language,popularity,similarity
0,5Jqbj4LWQxxjVTXtv2bd61,The Scientist,Coldplay,Family Time,2024,English,0,0.993599
1,75lxsM88TtWfBvyjjwJOIS,Jalani Hariku,Shakira Jasmine,Female Collection - Vol. 1,2005,English,4,0.983118
2,7dfjHtYzdCiZ934Hyxm2HM,A Big Train Is Comin',Madonnas in a Field,Standing On a Ridgeline,2012,English,0,0.980231
3,5RgSyslysUK17GhRli5ksZ,Outside,The Weeknd,Trilogy,2012,English,44,0.976960
4,6sQckd3Z8NPxVVKUnavY1F,'tis the damn season,Taylor Swift,evermore (deluxe version),2021,English,69,0.964485
5,7ugJ76kbyKYdDx0iee9S0d,If the World Ends,Madonnas in a Field,Standing On a Ridgeline,2012,English,0,0.961326
6,2jGGb3O8FNPcDAPh1d7Qdo,Rends mon coeur lilas,Clothilde Madonna,Rends mon coeur lilas,2022,English,0,0.955739
7,0ls2HeVRI2LELC9uk43Kje,With A Child's Heart,Michael Jackson,Music and Me,1973,English,24,0.954621
8,5cijobH65WeTnTi1f8yNFu,Something to Remember,Madonna,I'm Breathless,1990,English,26,0.952710
9,6EKDPhgCDL2cwRvloAEwSP,Name (feat. Tori Kelly),"Justin Bieber, Tori Kelly",Justice (Triple Chucks Deluxe),2021,English,48,0.948606


In [8]:
tar_hin = ["Hindi"]
df_hin = df[df["language"].isin(tar_hin)]

In [12]:
df_hin = df_hin.drop_duplicates(
    subset="track_id"
).copy()

print("After removing duplicate track IDs:", df_hin.shape)

After removing duplicate track IDs: (5740, 22)


In [14]:
df_hin = df_hin.drop_duplicates(
    subset=["track_name", "artist_name"],
    keep="first"
).copy()

print("Dataset shape after removing track + artist duplicates:", df_hin.shape)

Dataset shape after removing track + artist duplicates: (3918, 22)


In [15]:
df_hin = df_hin.dropna(
    subset=feature_columns
).copy()

print("After removing missing feature values:", df_hin.shape)

After removing missing feature values: (3918, 22)


In [18]:
df_hin = df_hin.reset_index(drop=True)

In [21]:
X_hin = df_hin[feature_columns].copy()

print("Feature matrix shape:", X_hin.shape)
display(X_hin.head())

Feature matrix shape: (3918, 9)


,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence
0,0.28000,0.684,0.772,0.00520,0.0818,-8.282,0.0419,106.031,0.451
1,0.02660,0.669,0.863,0.00000,0.3300,-3.364,0.1190,93.016,0.486
2,0.00069,0.641,0.782,0.65200,0.0767,-6.228,0.0456,93.006,0.376
3,0.74500,0.483,0.572,0.02470,0.0912,-11.780,0.0409,83.881,0.541
4,0.00442,0.511,0.683,0.00516,0.0890,-6.484,0.0427,123.850,0.254


In [25]:
scaler_hin = StandardScaler()

X_scaled_hin = scaler_hin.fit_transform(X_hin)

print("Scaled feature matrix shape:", X_scaled_hin.shape)

Scaled feature matrix shape: (3918, 9)


In [28]:
knn_hin = NearestNeighbors(
    n_neighbors=20,
    metric="cosine",
    algorithm="brute"
)

knn_hin.fit(X_scaled_hin)

print("KNN model for Hindi created successfully.")

KNN model for Hindi created successfully.


In [29]:
def recommend_from_features(
    acousticness,
    danceability,
    energy,
    instrumentalness,
    liveness,
    loudness,
    speechiness,
    tempo,
    valence,
    n_recommendations=5
):
    
    # Create a DataFrame containing the new song
    new_song = pd.DataFrame([{
        "acousticness": acousticness,
        "danceability": danceability,
        "energy": energy,
        "instrumentalness": instrumentalness,
        "liveness": liveness,
        "loudness": loudness,
        "speechiness": speechiness,
        "tempo": tempo,
        "valence": valence
    }])
    
    # Scale using the SAME scaler used during model preparation
    new_song_scaled = scaler_hin.transform(new_song[feature_columns])
    
    # Find nearest songs
    distances, indices = knn_hin.kneighbors(
        new_song_scaled,
        n_neighbors=n_recommendations
    )
    
    # Build result
    recommendations = df_hin.iloc[indices[0]].copy()
    
    # Convert cosine distance to cosine similarity
    recommendations["similarity"] = 1 - distances[0]
    
    # Select useful columns
    result_columns = [
        "track_id",
        "track_name",
        "artist_name",
        "album_name",
        "year",
        "language",
        "popularity",
        "similarity"
    ]
    
    return recommendations[result_columns].reset_index(drop=True)

In [31]:
recommendations = recommend_from_features(
    acousticness=0.79,
    danceability=0.535,
    energy=0.462,
    instrumentalness=2.07e-06,
    liveness=0.114,
    loudness=-11.291,
    speechiness=0.0296,
    tempo=145.959,
    valence=0.23,
    n_recommendations=10
)

display(recommendations)

,track_id,track_name,artist_name,album_name,year,language,popularity,similarity
0,0uBo93xl23O60oErtKvSAg,Mazaak,Anuv Jain,Mazaak,2022,Hindi,63,1.000000
1,5BDjTqqglzGrL0XXZUQBrY,"Dil Ke Paas (From ""Wajah Tum Ho"")","Arijit Singh, Tulsi Kumar, Neuman Pinto",Love Forever With Arijit Singh,2017,Hindi,39,0.991983
2,5PO4sN3MWXOeQLGurvEvqM,Dil Ke Paas,"Arijit Singh, Tulsi Kumar, Neuman Pinto",Wajah Tum Ho,2016,Hindi,42,0.991983
3,2yUJZywLg2rJ2z5xDRGfAh,"Maan Le (From ""Chitrakut"")","Arijit Singh, Somesh Saha","Maan Le (From ""Chitrakut"")",2022,Hindi,18,0.976651
4,3I98QafBoaymQEiqhQRcqh,Maan Le,"Arijit Singh, Somesh Saha",Chitrakut (Original Motion Picture Soundtrack),2022,Hindi,27,0.976651
5,6etM3E9XDnrCyzXcI7whH7,Taron Ke Us Desh,"Javed Ali, Lata Bardoloi, Sumit Acharya",Taron Ke Us Desh,2017,Hindi,0,0.973584
6,1ffXHBlD0bJR73pa5oO4LK,Husn,Anuv Jain,Sajni Aur Saajan - Love Songs,2024,Hindi,23,0.967541
7,2lNepYUscS3QZIHx2MHV86,"O Meri Jaan (From ""Tum Mile"")","Pritam, KK",Musical Bond: Pritam & KK,2015,Hindi,24,0.966541
8,1qvoTEJr6bCdSl0l6CuxEz,Kahani (Sonu's Version),"Pritam, Sonu Nigam, Amitabh Bhattacharya",Laal Singh Chaddha,2022,Hindi,33,0.965905
9,382emarYcsDvic5RCKaglv,"Kahani (Sonu's Version) [From ""Laal Singh Chad...","Pritam, Sonu Nigam, Amitabh Bhattacharya","Kahani (Sonu's Version) [From ""Laal Singh Chad...",2022,Hindi,34,0.965905


In [ ]:
'acousticness': 0.79, 'danceability': 0.535, 'energy': 0.462, 'instrumentalness': 2.07e-06, 'key': 2, 'liveness': 0.114, 'loudness': -11.291, 'mode': 1, 'speechiness': 0.0296, 'tempo': 145.959, 'valence': 0.23}

In [38]:
def recommend_from_dict_hin(
    song_features,
    n_recommendations=10
):
    
    # Check that all required features exist
    missing_features = [
        feature
        for feature in feature_columns
        if feature not in song_features
    ]
    
    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )
    
    # Create input DataFrame
    new_song = pd.DataFrame(
        [{
            feature: song_features[feature]
            for feature in feature_columns
        }]
    )
    
    # Check for missing values
    if new_song.isnull().any().any():
        raise ValueError(
            "Input contains missing values."
        )
    
    # Scale
    new_song_scaled = scaler_hin.transform(
        new_song
    )
    
    # Find nearest songs
    distances, indices = knn_hin.kneighbors(
        new_song_scaled,
        n_neighbors=n_recommendations
    )
    
    # Get songs
    recommendations = df_hin.iloc[
        indices[0]
    ].copy()
    
    # Cosine similarity
    recommendations["similarity"] = (
        1 - distances[0]
    )
    
    # Output columns
    result_columns = [
        "track_id",
        "track_name",
        "artist_name",
        "album_name",
        "year",
        "language",
        "popularity",
        "similarity"
    ]
    
    return recommendations[
        result_columns
    ].reset_index(drop=True)

In [39]:
song_features = {
    "acousticness": 0.18,
    "danceability": 0.76,
    "energy": 0.91,
    "instrumentalness": 0.00,
    "liveness": 0.10,
    "loudness": -4.2,
    "speechiness": 0.05,
    "tempo": 168.0,
    "valence": 0.82
}

In [41]:
recommend_from_dict_hin(song_features, n_recommendations=2)

,track_id,track_name,artist_name,album_name,year,language,popularity,similarity
0,0rH5gUdg4f4x3ez7zvWjJR,Ho Gayi Tun,"Pritam, Yashita Yashpal, Bob",Players,2011,Hindi,25,0.944438
1,0Z2wPcM35fgU3uLh9GzgRf,Lalaten Jara Ke,"Kumar Pritam, Suman Gupta",Lalaten Jara Ke,2021,Hindi,28,0.938889


In [43]:
import joblib

joblib.dump(
    scaler_hin,
    "scaler_hin.pkl"
)

joblib.dump(
    knn_hin,
    "knn_model_hin.pkl"
)

df_hin.to_pickle(
    "hin_song_catalog.pkl"
)

print("Model files saved successfully.")

Model files saved successfully.


In [44]:
joblib.dump(
    scaler_eng,
    "scaler_eng.pkl"
)

joblib.dump(
    knn_eng,
    "knn_model_eng.pkl"
)

df_eng.to_pickle(
    "eng_song_catalog.pkl"
)

print("Model files saved successfully.")

Model files saved successfully.


# Testing

In [12]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

CLIENT_ID = "436f9432173944fd8b76bc1572ea98c0"
CLIENT_SECRET = "9fc27fc1847845d29a2d958c2115513f"


def get_access_token():
    url = "https://accounts.spotify.com/api/token"

    response = requests.post(
        url,
        data={"grant_type": "client_credentials"},
        auth=(CLIENT_ID, CLIENT_SECRET)
    )

    response.raise_for_status()

    return response.json()["access_token"]


TOKEN = get_access_token()

print("Spotify authentication successful!")

Spotify authentication successful!


In [13]:
import requests

def get_track_id(song_name, token):
    search_url = "https://api.spotify.com/v1/search"
    
    headers = {
        "Authorization": f"Bearer {token}"
    }
    
    params = {
        "q": song_name,
        "type": "track",
        "limit": 1
    }
    
    response = requests.get(
        search_url,
        headers=headers,
        params=params
    ).json()
    
    try:
        return response["tracks"]["items"][0]["id"]
    except (KeyError, IndexError):
        print("Song not found.")
        return None


def get_reccobeats_id(spotify_track_id):
    url = "https://api.reccobeats.com/v1/track"

    params = {
        "ids": spotify_track_id
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("ReccoBeats Track Error:", response.status_code)
        print(response.text)
        return None

    data = response.json()

    try:
        return data["content"][0]["id"]
    except (KeyError, IndexError):
        print("Track not found in ReccoBeats.")
        return None


def get_audio_features(reccobeats_id):
    url = f"https://api.reccobeats.com/v1/track/{reccobeats_id}/audio-features"

    response = requests.get(url)

    if response.status_code != 200:
        print("Audio Features Error:", response.status_code)
        print(response.text)
        return None

    return response.json()

In [16]:
x = get_track_id("Gerua", TOKEN)
print(x)
y = get_reccobeats_id(x)
print(get_audio_features(y))

7iLA6PQeQWRjLDHOJgGpj5
{'id': '15841412-7495-40f6-bb86-dee5cef22a58', 'href': 'https://open.spotify.com/track/7iLA6PQeQWRjLDHOJgGpj5', 'isrc': 'INS171501741', 'acousticness': 0.636, 'danceability': 0.371, 'energy': 0.668, 'instrumentalness': 2e-06, 'key': 11, 'liveness': 0.294, 'loudness': -7.269, 'mode': 0, 'speechiness': 0.0497, 'tempo': 87.458, 'valence': 0.403}


In [15]:
y = get_reccobeats_id("0AxJl3J0idAYqxZtgICDPC")
print(get_audio_features(y))

{'id': '185e207e-8ac9-477b-ba68-7bc37e0a4c26', 'href': 'https://open.spotify.com/track/0AxJl3J0idAYqxZtgICDPC', 'isrc': 'INS171501741', 'acousticness': 0.636, 'danceability': 0.371, 'energy': 0.668, 'instrumentalness': 2e-06, 'key': 11, 'liveness': 0.294, 'loudness': -7.269, 'mode': 0, 'speechiness': 0.0497, 'tempo': 87.458, 'valence': 0.403}


In [ ]:
import pickle
import numpy as np

# 1. Load the model from the .pkl file
with open('model.pkl', 'rb') as file:
    model = pickle.load(file)

In [5]:
import pandas as pd
import joblib

# -----------------------------
# Load saved model files
# -----------------------------
scaler = joblib.load("scaler_eng.pkl")
knn = joblib.load("knn_model_eng.pkl")
df = pd.read_pickle("eng_song_catalog.pkl")

# Features used by the model
feature_columns = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence"
]

# -----------------------------
# Input: features of ANY song
# -----------------------------
song_features = {
    "acousticness": 0.18,
    "danceability": 0.76,
    "energy": 0.91,
    "instrumentalness": 0.00,
    "liveness": 0.10,
    "loudness": -4.2,
    "speechiness": 0.05,
    "tempo": 168.0,
    "valence": 0.82
}

n_recommendations = 2

# -----------------------------
# Prepare input
# -----------------------------
new_song = pd.DataFrame([song_features])[feature_columns]

# Scale using the saved scaler
new_song_scaled = scaler.transform(new_song)

# -----------------------------
# Get recommendations
# -----------------------------
distances, indices = knn.kneighbors(
    new_song_scaled,
    n_neighbors=n_recommendations
)

recommendations = df.iloc[indices[0]].copy()

# Convert cosine distance to similarity
recommendations["similarity"] = 1 - distances[0]

# -----------------------------
# Display results
# -----------------------------
result_columns = [
    "track_id",
    "track_name",
    "artist_name",
    "album_name",
    "year",
    "language",
    "popularity",
    "similarity"
]

result_columns = [
    col for col in result_columns
    if col in recommendations.columns
]

recommendations = recommendations[result_columns]

display(recommendations)

,track_id,track_name,artist_name,album_name,year,language,popularity,similarity
13508,5AmxRmimGnJS41AyCe22lX,Omwoyogwo,Nampijja Shakirah,Omusajja Alunjiwa Sente,2015,English,0,0.985786
7776,6aiuOkaPgVROiE98FhCqKI,Hanky Panky - Bare Bones Single Mix,"Madonna, Kevin Gilbert",Hanky Panky,1990,English,23,0.984275
